# Quotes Scraper

Scrapes quotes, authors, and tags from [quotes.toscrape.com](https://quotes.toscrape.com) (a site built for scraping practice) across 10 paginated pages, and exports the result to `data2.csv`.

In [1]:
import os
import requests
from bs4 import BeautifulSoup
import pandas as pd

BASE_URL = "https://quotes.toscrape.com/page/{}/"
HTML_DIR = "html2"
NUM_PAGES = 10

os.makedirs(HTML_DIR, exist_ok=True)

In [2]:
def scrape_quotes(base_url, num_pages, html_dir):
    items = []
    for i in range(1, num_pages + 1):
        path = f"{html_dir}/page{i}.html"

        # Download (skip if already saved)
        if not os.path.exists(path):
            try:
                response = requests.get(base_url.format(i), timeout=10)
                response.raise_for_status()
            except requests.RequestException as e:
                print(f"Failed to fetch page {i}: {e}")
                continue
            with open(path, 'w', encoding="utf-8") as f:
                f.write(response.text)

        # Parse
        with open(path, 'r', encoding="utf-8") as f:
            soup = BeautifulSoup(f.read(), 'html.parser')

        for article in soup.select(".quote"):
            try:
                quote = article.select_one(".text").text
                author = article.select_one(".author").text
                tags = [tag.get_text(strip=True) for tag in article.select("a.tag")]
                items.append([quote, author, tags, i])
            except AttributeError as e:
                print(f"Skipped a malformed entry on page {i}: {e}")
    return items

items = scrape_quotes(BASE_URL, NUM_PAGES, HTML_DIR)
df = pd.DataFrame(items, columns=["quote", "author", "tags", "page"])
print(f"Scraped {len(df)} quotes.")
df.head()

Scraped 100 quotes.


,quote,author,tags,page
0,“The world as we have created it is a process ...,Albert Einstein,"[change, deep-thoughts, thinking, world]",1
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling,"[abilities, choices]",1
2,“There are only two ways to live your life. On...,Albert Einstein,"[inspirational, life, live, miracle, miracles]",1
3,"“The person, be it gentleman or lady, who has ...",Jane Austen,"[aliteracy, books, classic, humor]",1
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,"[be-yourself, inspirational]",1


In [3]:
df.to_csv("data/data2.csv", index=False)
print("Saved to data/data2.csv")

Saved to data/data2.csv
